<a href="https://colab.research.google.com/github/SubhasishSahu/Avito/blob/master/Kasha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install streamlit -q
!pip install pyngrok

In [11]:
!pip install -q streamlit pandas numpy plotly yfinance openpyxl pyngrok

In [12]:
%%writefile app.py
import streamlit as st
import pandas as pd
import yfinance as yf
import plotly.express as px

st.set_page_config(page_title="Finance Simulator - Learning", layout="wide")

st.title("My Finance Modeling Tool (Prototype)")

st.markdown("""
Upload your stock holdings Excel/CSV (columns: Ticker, Shares, etc.)
Then see basics: current prices, simple portfolio value, sector pie.
More features coming: fundamentals, simulation, news impact...
""")

# File upload
uploaded = st.file_uploader("Upload holdings file", type=["xlsx", "xls", "csv"])

if uploaded is not None:
    if uploaded.name.endswith('.csv'):
        df = pd.read_csv(uploaded)
    else:
        df = pd.read_excel(uploaded)

    st.subheader("Your Holdings")
    st.dataframe(df)

    if 'Ticker' in df.columns:
        # Fetch current data
        tickers = df['Ticker'].unique().tolist()
        data = yf.download(tickers, period="1d")['Adj Close'].iloc[-1]
        df['Current Price'] = df['Ticker'].map(data)
        df['Value'] = df['Shares'] * df['Current Price']  # assume 'Shares' column

        total_value = df['Value'].sum()
        st.metric("Portfolio Value", f"€{total_value:,.2f}")

        # Simple pie chart by ticker (or add 'Sector' column later)
        fig = px.pie(df, values='Value', names='Ticker', title='Portfolio Allocation')
        st.plotly_chart(fig, use_container_width=True)

        st.subheader("Quick Fundamentals (example for first ticker)")
        if tickers:
            info = yf.Ticker(tickers[0]).info
            st.json({k: info.get(k) for k in ['longName', 'sector', 'trailingPE', 'forwardPE', 'marketCap']})
else:
    st.info("Upload a file to start → example format: Ticker | Shares | Sector (optional)")

Overwriting app.py


In [13]:
from pyngrok import ngrok

# Replace with YOUR authtoken
!ngrok config add-authtoken 3ABDlUCKwcUpXGvEusJbE4SGxMo_2WyB86j722KfsE7Gk2C3h

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [14]:
# Start Streamlit in background
!streamlit run app.py &>/dev/null&

# Create tunnel → copy the https://xxxx.ngrok-free.app URL
public_url = ngrok.connect(8501)
print("Your Streamlit app is live at:", public_url)

Your Streamlit app is live at: NgrokTunnel: "https://shira-subaffluent-kasha.ngrok-free.dev" -> "http://localhost:8501"
